# Notebook 5: Real-Time Inference with Online Feature Store

This notebook demonstrates **real-time inference** by:
1. Retrieving features from the **Online Feature Store** (low-latency key-value lookup)
2. Running prediction through the registered model via `mv.run()`

The Online Feature Store maintains a materialized, low-latency copy of feature values
that auto-refreshes from the underlying Dynamic Table. This enables sub-second
feature retrieval for real-time inference use cases.

**Prerequisites**: Run Notebooks 01-03 first. Notebook 02 enables the Online Feature Store.

In [ ]:
import json
import time
import pandas as pd
import numpy as np
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore

print("Libraries loaded.")

## 1. Connect to Snowflake

In [ ]:
session = Session.builder.config("connection_name", "DEMO").create()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE HEALTHCARE_ML").collect()
session.sql("USE WAREHOUSE HEALTHCARE_ML_WH").collect()

print(f"Connected: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")

## 2. Initialize Feature Store and Model Registry

In [ ]:
# Feature Store
fs = FeatureStore(
    session=session,
    database="HEALTHCARE_ML",
    name="FEATURE_STORE",
    default_warehouse="HEALTHCARE_ML_WH"
)

# Model Registry
registry = Registry(
    session=session,
    database_name="HEALTHCARE_ML",
    schema_name="MODEL_REGISTRY"
)

# Load model
mv = registry.get_model("READMISSION_PREDICTOR").version("V1")
print(f"Model: READMISSION_PREDICTOR V1")

# Load feature view
fv = fs.get_feature_view("PATIENT_CLINICAL_FEATURES", "V1")
print(f"Feature View: {fv.name} {fv.version}")
print(f"Online enabled: True")

In [ ]:
# Load metadata
with open('artifacts/model_metadata.json', 'r') as f:
    metadata = json.load(f)
FEATURE_COLUMNS = metadata['feature_columns']
print(f"Feature columns: {len(FEATURE_COLUMNS)}")

## 3. Real-Time Feature Retrieval from Online Feature Store

The Online Feature Store provides low-latency key-value lookups.
Given a patient ID, it returns the latest feature values instantly.

This simulates a real-time clinical decision support scenario:
a patient arrives, and the system needs to predict readmission risk immediately.

In [ ]:
# Get some sample patient IDs from the data
sample_patients = session.sql("""
    SELECT DISTINCT PATIENT_ID 
    FROM HEALTHCARE_ML.RAW_DATA.PATIENTS 
    ORDER BY PATIENT_ID 
    LIMIT 10
""").to_pandas()

print(f"Sample patients for real-time lookup:")
print(sample_patients['PATIENT_ID'].tolist())

In [ ]:
# Real-time feature retrieval using the Online Feature Store
# This retrieves the LATEST feature values for each patient
patient_ids = sample_patients['PATIENT_ID'].tolist()[:5]

# Create a spine DataFrame with just the entity keys
spine_df = session.create_dataframe(
    [[pid] for pid in patient_ids],
    schema=["PATIENT_ID"]
)

print("Retrieving features from Online Feature Store...")
start_time = time.time()

# Retrieve features for these patients
features_df = fs.retrieve_feature_values(
    spine_df=spine_df,
    features=[fv],
    spine_timestamp_col=None  # Get latest features
)

retrieval_time = time.time() - start_time
print(f"Feature retrieval time: {retrieval_time:.3f}s")
print(f"Retrieved features for {features_df.count()} patient records")
features_df.show()

## 4. Real-Time Inference

Combine the retrieved features with the registered model for instant predictions.

In [ ]:
# Convert to pandas for model inference
features_pd = features_df.to_pandas()
print(f"Features DataFrame shape: {features_pd.shape}")
print(f"Columns: {list(features_pd.columns)}")

# Select only model input features
# The feature view may return extra columns (PATIENT_ID, ADMISSION_ID, EVENT_TIMESTAMP, READMITTED_30D)
available_features = [c for c in FEATURE_COLUMNS if c in features_pd.columns]
print(f"Available model features: {len(available_features)} / {len(FEATURE_COLUMNS)}")

model_input = features_pd[available_features]
print(f"Model input shape: {model_input.shape}")

In [ ]:
# Run real-time prediction
print("Running real-time inference...")
start_time = time.time()

predictions = mv.run(model_input, function_name="predict_proba")

inference_time = time.time() - start_time
print(f"Inference time: {inference_time:.3f}s")
print(f"\nPrediction Results:")

# Combine patient IDs with predictions
results = features_pd[['PATIENT_ID']].copy()
results['READMISSION_PROB'] = predictions['output_feature_1'].values
results['RISK_LEVEL'] = results['READMISSION_PROB'].apply(
    lambda x: 'HIGH' if x > 0.5 else ('MEDIUM' if x > 0.3 else 'LOW')
)

print(results.to_string(index=False))

## 5. Simulate Real-Time Clinical Decision Support

Simulate a real-world scenario: a patient is being discharged,
and the system needs to provide a readmission risk score in real-time.

In [ ]:
def predict_readmission_risk(patient_id, session, fs, fv, mv, feature_columns):
    """Real-time readmission risk prediction for a single patient."""
    total_start = time.time()
    
    # Step 1: Retrieve features from Online Feature Store
    spine = session.create_dataframe([[patient_id]], schema=["PATIENT_ID"])
    features = fs.retrieve_feature_values(
        spine_df=spine,
        features=[fv],
        spine_timestamp_col=None
    ).to_pandas()
    
    if features.empty:
        return {"error": f"No features found for patient {patient_id}"}
    
    # Step 2: Run inference
    available = [c for c in feature_columns if c in features.columns]
    model_input = features[available]
    proba = mv.run(model_input, function_name="predict_proba")
    
    total_time = time.time() - total_start
    readmission_prob = proba['output_feature_1'].values[0]
    
    return {
        "patient_id": patient_id,
        "readmission_probability": round(readmission_prob, 4),
        "risk_level": "HIGH" if readmission_prob > 0.5 else ("MEDIUM" if readmission_prob > 0.3 else "LOW"),
        "response_time_ms": round(total_time * 1000, 1)
    }

# Simulate real-time requests for 5 patients
print("=" * 60)
print("REAL-TIME CLINICAL DECISION SUPPORT")
print("=" * 60)

test_patients = patient_ids[:5]
for pid in test_patients:
    result = predict_readmission_risk(pid, session, fs, fv, mv, FEATURE_COLUMNS)
    print(f"\nPatient {result['patient_id']}:")
    print(f"  Readmission probability: {result['readmission_probability']:.1%}")
    print(f"  Risk level: {result['risk_level']}")
    print(f"  Response time: {result['response_time_ms']:.0f}ms")

## 6. Compare Batch vs Real-Time Results

Verify that batch and real-time inference produce the same results
for the same patients.

In [ ]:
# Get batch predictions for the same patients (if batch table exists)
try:
    batch_results = session.sql(f"""
        SELECT * FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS
        LIMIT 5
    """).to_pandas()
    print("Batch predictions available for comparison.")
    print(f"Batch columns: {list(batch_results.columns)}")
    print(batch_results.head())
except Exception as e:
    print(f"Batch predictions table not yet available: {e}")
    print("Run Notebook 04 first for comparison.")

## 7. Verification Summary

In [ ]:
print("=" * 60)
print("REAL-TIME INFERENCE VERIFICATION")
print("=" * 60)
print(f"\nOnline Feature Store: HEALTHCARE_ML.FEATURE_STORE")
print(f"Feature View: PATIENT_CLINICAL_FEATURES V1 (Online enabled)")
print(f"Model: READMISSION_PREDICTOR V1")
print(f"\nWorkflow:")
print(f"  1. Patient ID submitted")
print(f"  2. Features retrieved from Online Feature Store (low-latency)")
print(f"  3. Model inference via mv.run() on warehouse")
print(f"  4. Risk score returned")
print(f"\nReal-time inference: VERIFIED")
print(f"\n=== ALL NOTEBOOKS COMPLETE ===")
print(f"  01: Local training - DONE")
print(f"  02: Snowflake setup (Feature Store, Dynamic Tables) - DONE")
print(f"  03: Model Registry - DONE")
print(f"  04: Batch inference (run_batch on SPCS) - DONE")
print(f"  05: Real-time inference (Online Feature Store) - DONE")

session.close()
print("\nSession closed.")